
# ✈️ Final Project: Predicting Flight Departure Delays
> **⚠️ Important:** Images may not render directly in this notebook.  
> Please **click the image links in each Markdown cell** to view the notebook visualizations.

## 👤 Phase Leader Plan

| Phase   | Leader       | Dates      | Responsibilities                                 |
|---------|--------------|------------|--------------------------------------------------|
| Phase 1 | Luc Rieffel  | July 1-13  | Abstract, Credit Assignment Plan, Data Description, Notebook Inspection & Submission |
| Phase 2 |  ()| July 14-27  | |
| Phase 3 |  ()| July 28-Aug 3  | |
| Phase 4|  ()| Aug 4-9  | |

## 👥 Credit Assignment Plan

| Team Member        | Responsibilities                                                                 |
|--------------------|----------------------------------------------------------------------------------|
| Luc Rieffel        | Abstract, data description, data cleaning, feature engineering, ML & data pipeline setup, feature selection         |
| Noah Lomnitz       | Exploratory Data Analysis (EDA), summary visuals               |
| Alexander Caichen  | Machine learning algorithms, evaluation metrics                         |
| Ronald Nap         | Pipeline, Block Diagrams, Gantt Chart  


##Gantt Chart 
**PLEASE CLICK THE LINKS TO VIEW THE IMAGES, DATABRICKS WON'T RENDER THE IMAGES**

[Gannt Chart](https://github.com/luc-rieffel-berkeley/datasets/blob/main/timeline.png)


<!-- ![Timeline](/Workspace/Users/rnap@berkeley.edu/figures/timeline.png) -->


## Abstract

The aim of our project is to predict whether a flight will be delayed by more than 15 minutes using flight schedule, weather, and airport metadata from the On-Time Performance and Weather (OTPW) dataset. By leveraging the 3-month pre-joined subset OTPW dataset for development and evaluation, we aim to produce a model with the potential to reduce overhead costs for airlines, improve the customer experience, and identify causal flight delay indicators to improve operational efficiency.

Preliminary exploratory data analysis (EDA) reveals strong seasonal and time-of-day patterns in delays, as well as low/moderate relationship between delay time and weather features such as visibility, precipitation, and wind speed. We handle missing and inconsistent data through standard cleaning, mean imputation, and categorical encoding, followed by scaling of numeric features.

In future stages, we plan to build a Spark ML pipeline using classification algorithms such as logistic regression as a baseline, followed by more expressive models like gradient boosted trees. All models will be evaluated using consistent splits (60% train / 20% validation / 20% test) and Spark's scalable ML pipeline tooling. We designed an appropriate modeling, data cleaning, and cross validation pipeline to remove the potential for data leakage.

Our model will be a binary classifier outputting the probability that a flight will be delayed by 15 or more minutes (with a threshold of 50% used to classify the outcome). We will use logistic loss with L1 regularization to prevent overfitting. Model performance will be assessed using accuracy, precision, recall, and F1 score to ensure robustness under different operational tradeoffs.

This project addresses a practical and operationally significant problem. Airlines, airports, and third-party travel apps could deploy the model to proactively notify passengers and adjust gate, staffing, or runway plans. In the real world, constraints such as interpretability, inference time, and the ability to refresh the model daily or hourly must be taken into account when evaluating final model deployment.

## 📊 Data Description

- **Sources**: On-Time Performance (Flight) Data, Hourly Weather Data, and Station Metadata from `dbfs:/mnt/mids-w261/OTPW_3M_2015.csv`
- **Date Range**: 01/01/2015 to 03/31/2015
- **Format**: Pre-joined Parquet files, joined by flight date/time and location (origin/destination)
- **Size(before filtering)**: ~1.4M records, 200+ columns before filtering. 
- **Size (after filtering & cleaning)**: ~1.35M rows, 26 columns (including multiple outcome variables we will test models with later) 
- **Size (after feature engineering)**: ~1.35M rows, 39 columns (includes OHE categorical features)
- Checkpointing Strategy: 
  - Run through EDA, Data Cleaning, and Feature Engineering
  - Save each checkpoint as a parquet file at the end of the notebook
  - Set `override = True` to ensure the files update when we re-run the code to make changes
  - If we don't make changes to our pipeline, we don't have to re-run all of our steps to read in data and continue 

---

**Data Cleaning & Filtering Strategy**:
  - Dropped cancelled/diverted flights(removed ~3% of the dataset)
  - Removed rows with missing outcome variables 
  - 
  - Imputed missing numerical features with the mean values(there were sufficient sample sizes of 3k+ rows per feature)
  - There were no null categorical features after cleaning

  <!-- - Imputed missing categorical features with the MODE -->
  - Cast numeric features to `DoubleType`
  - Dropped **duplicate** rows based on any duplicate combinations in which rows of all selected features + outcome variables were exactly the same (616 duplicate rows)
  
- **Feature Engineering Strategy**:
  - Categorical Features: Apply one-hot encoding
  - Numerical Features: Use standard scaler

- **Train/Test/Validation Split**
  - 60% training data
  - 20% validation data
  - 20% testing data
  - All features that could introduce data leakage were removed
  - Standard Scaler and other normalization methods were applied only to the training data

- **Feature Selection Strategy**:
  - Our goal was to choose features in each category (flight info, weather, airport info) that intuitively appeared to influence DEP_DELAY.
  - Our initial approach was to use intuition. However, in the later model building stages we plan to create a correlation plot of the most correlated features with our outcome variable. If we continue to see performance issues we can also look at any correlations between features and post-flight information such a `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY` or `LATE_AIRCRAFT_DELAY` to extract causal information about the source of the flight delay.
  - Our initial scope is binary classification with the range Delay > 15 minutes however we may change this later to balance the model's real-world usability and interpretability. 



# Exploratory Data Analysis
We investigated whether **weather conditions 2 hours before departure** at both origin and destination airports influence departure delays.

## Visibility & Wind Speed
- **Departure Airport**: Visibility and wind speed showed **no strong relationship** to delays.
- **Arrival Airport**: Slightly **fewer delays** when visibility >10 miles. Wind speed again showed **minimal effect**.
*<!-- Insert visibility & wind speed charts here -->*

  
[EDA Graph 1: Hourly Visibility(click here)](https://imgur.com/JMFPY8N)

[EDA Graph 2: Hourly Wind Speed Boxplot(click here)](https://imgur.com/Tv5dpQo)



## Airline-Level Differences
- **Median delays** were consistent across airlines.
- Some variation at the **75th percentile** suggests potential differences in delay variability.
[EDA Graph 3: Departure Delay by Top 5 Carrier(click here)](https://imgur.com/2fxJtZX)
*<!-- Insert airline delay distribution chart here -->*

## Takeaways
- **Weather 2 hours prior** is not a strong predictor overall.
- May explore **airport-specific patterns** and other more predictive features going forward.
- **Airline ID** could be useful as a categorical feature given the delay airline level differences highlighted in graph 3. 

## 📚 Data Dictionary

### ✈️ Flight Schedule & Route Features
| Feature               | Description                                                       |
|------------------------|-------------------------------------------------------------------|
| `DISTANCE`             | Flight distance in miles                                          |
| `CRS_ELAPSED_TIME`     | Scheduled flight duration in minutes                              |
| `DEP_TIME_BLK`         | Scheduled departure time block (e.g., 0600-0659)                  |
| `MONTH`                | Month of departure (1-12)                                         |
| `DAY_OF_MONTH`         | Day of the month (1-31)                                           |
| `DAY_OF_WEEK`          | Day of the week (1=Monday, 7=Sunday)                              |

### 🌦️ Weather Features
| Feature                   | Description                                                   |
|----------------------------|---------------------------------------------------------------|
| `HourlyDryBulbTemperature` | Ambient temperature (F), unaffected by moisture              |
| `HourlyDewPointTemperature`| Dew point temperature (F), indicates air moisture            |
| `HourlyPrecipitation`      | Precipitation in inches                                       |
| `HourlyWindSpeed`          | Wind speed in mph                                             |
| `HourlyVisibility`         | Visibility in miles                                           |
| `HourlyRelativeHumidity`   | Relative humidity (%)                                         |
| `HourlySeaLevelPressure`   | Atmospheric pressure at sea level                             |
| `HourlyAltimeterSetting`   | Altimeter setting for aircraft instruments                    |

### 🛫 Airport Information
| Feature         | Description                                          |
|------------------|------------------------------------------------------|
| `origin_type`    | Origin airport type (e.g., large_airport)            |
| `dest_type`      | Destination airport type                             |
| `origin_region`  | Region of origin airport (e.g., US-CA, US-NY)        |
| `dest_region`    | Region of destination airport                        |

### 🎯 Outcome Variable
| Outcome Variable    | Description                                                      |
|---------------------|------------------------------------------------------------------|
| `DEP_DEL15`         | Binary: 1 if departure delay > 15 mins, else 0            |

### 🎯  Other Potential Outcome Variables (We plan to experiment with these later)
| Outcome Variable    | Description                                                      |
|---------------------|------------------------------------------------------------------|
| `DEP_DELAY`         | Actual departure delay in minutes (negative = early departure)   |
| `DEP_DELAY_NEW`     | Same as `DEP_DELAY` but negatives set to 0                       |
| `DEP_DELAY_GROUP`   | Delay grouped into bins (e.g., -1, 0, 1-14, 15-29 mins)           |

<!-- | `CRS_DEP_TIME`         | Scheduled departure time in HHMM format                          | -->


# Machine Learning Pipeline
<!-- ![Fig 1](/Workspace/Users/rnap@berkeley.edu/figures/fig1.png) -->

[Machine Learning Pipelines Diagram(click here)](https://github.com/luc-rieffel-berkeley/datasets/blob/main/fig1.png)


# Model Training and Evaluation
### Model:

To start off, we plan on building a classification model rather than a regression model. This is because we first want to develop a proof-of-concept to see if our features can even be used to effectively predict variables related to plane-lateness. The model will be based on gradient descent, with weight `θ` subtracted by the learning rate `α` multiplied by the gradient of the cost function `∇J(θ)` during each step of training to obtain the updated weight values.
$$\boldsymbol{\theta}_{n+1} = \boldsymbol{\theta}_n - \alpha \nabla J(\boldsymbol{\theta_n})$$

Since we are building a binary classifier we will be using Accuracy as the loss metric instead of Mean Squared Error. For the regularization method we will start off by using L1, or Lasso Regression, giving us the following equation for our cost function `J(θ)`:
$$J(\mathbf{w}) = - {Accuracy} + \lambda \sum_{j=1}^n |w_j| = - \frac{TP + TN}{TP + TN + FP + FN} + \lambda \sum_{j=1}^n |w_j|$$
Lasso Regression is chosen for its tendency to aggresively reduce collinear feature weights. We suspect several, though not a majority, of features may be correlated, such as `HourlyDewPointTemperature` with `HourlyRelativeHumidity`, both of which are affected by moisture in the air, and would like to prevent such redundant features from influencing model accuracy. Additionally, we would like to identify such collinear features via the presence of low weights after training to optimize the training data.


### Evaluation:

Accuracy, Precision, Recall, and the F1 score will be used as evaluation metrics for our binary classifier model.

1. For each sample, add together the model weight multiplied by the corresponding value. In other words:
    $$z = \sum_{i=1}^{n} w_i x_i$$
    where `w` is the model weight, `x` is the corresponding feature value from the sample, and `i` is the sample number.
2. Apply a sigmoid function to the output of the sums then apply `> 0.5` to that output, giving a True/False value representing whether the model predicts a flight will be at least 15 minutes late based on the sample's data.
    * Reminder that the sigmoid function is 
    $$\sigma(x) = \frac{1}{1 + e^{-x}}$$
    where `x` is the summation output, and returns a values between 0 and 1 (representing the probability of being late by more than 15 minutes in this case) no matter the input value.
4. Compare the resulting boolean to the `DEP_DEL15` truth value to determine if the model prediction is a true positive (TP), true negative (TN), false positive (FP), or false negative (FN). Results are stored in an array of 4 values (`(FP,TN,FP,FN)`), where only one value is 1 and the rest are zero (similar to one hot encoding but for true positive, negative, etc. labels).
5. Apply a reducer to the tuple from the previous step, adding them all together, giving the total number of true positive, true negative, false positive, and false negative predictions from the data set.
6. Equations for each metric:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$
$$
\text{Precision} = \frac{TP}{TP + FP}
$$
$$
\text{Recall} = \frac{TP}{TP + FN}
$$
$$
\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

# References:

## Code Notebook Link: 
https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/1422956527184026?o=4021782157704243


# Know your mount
Here is the mounting for this class, your source for the original data! Remember, you only have Read access, not Write! Also, become familiar with `dbutils` the equivalent of `gcp` in DataProc